> **Meeting 7, as we went through it in class.** This is the notebook on the
> screen, with the instructor's notes removed. Your own notebook for the meeting —
> the one with the studio — is in the same folder.

# STAT 764
## Regularization

### Meeting 7 — Tuesday, September 29

Least squares has no brake. Hand it enough columns and it will fit anything — including
noise — and give you back coefficients, p-values and a training R-squared that all look
like findings.

That is not a hypothetical. It is how a good share of published regressions were built,
and it is what you get from any tool that picks predictors by staring at the same data
it then reports on.

Today:

1. **A regression worth reporting** — until you find out what it was fitted to.
2. **What happens with no brake**, measured on Ames with 319 columns.
3. **What a penalty does**, and why ridge and lasso behave differently.
4. **Why the penalty cares about your units** when least squares mostly does not.

In [ ]:
# Setup. Run before they walk in.
import os
import pathlib
import sys

here = pathlib.Path.cwd()
cands = []
if os.environ.get("STAT764_REPO"):
    cands.append(pathlib.Path(os.environ["STAT764_REPO"]))
cands += [p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
cands += [c.parent.parent for c in here.glob("*/course/stat764.py")]
cands += [p / "stat764-fall2026" for p in [here, *here.parents]]
cfg = next((p / "course.yaml" for p in [here, *here.parents]
            if (p / "course.yaml").exists()), None)
if cfg:
    for line in cfg.read_text().splitlines():
        if line.startswith("student_repo:"):
            cands.append(pathlib.Path(line.split(":", 1)[1].strip()))

repo = next((c for c in cands if (c / "course" / "stat764.py").exists()), None)
if repo is None:
    raise RuntimeError(
        "Cannot find the course files.\n"
        f"  looked from: {here}\n"
        "  Fix: set STAT764_REPO, or run from a folder with course/stat764.py beside it.")
os.environ["STAT764_REPO"] = str(repo)
sys.path.insert(0, str(repo / "course"))

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from stat764 import load

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# A 300-house sample often contains no house with a pool, so Pool_QC is entirely
# missing and the imputer (correctly) skips it -- with a warning on every fit.
# Silence that one message and nothing else.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")

ames = load("ames.csv")
y = ames["SalePrice"]

# Every column except the two ID columns and the answer.
NUMERIC = [c for c in ames.select_dtypes("number").columns
           if c not in ("Order", "PID", "SalePrice")]
CATEGORICAL = ames.select_dtypes(exclude="number").columns.tolist()


def pipeline(model=None, numeric=NUMERIC, categorical=CATEGORICAL, scale=True):
    """The Meeting 2 skeleton. `scale=False` drops the StandardScaler, for Section 4."""
    num = [("fill", SimpleImputer(strategy="median"))]
    if scale:
        num.append(("scale", StandardScaler()))
    parts = [("num", Pipeline(num), numeric)]
    if categorical:
        parts.append(("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                              fill_value="Missing")),
                                       ("encode", OneHotEncoder(handle_unknown="ignore"))]),
                      categorical))
    return Pipeline([("prep", ColumnTransformer(parts)),
                     ("model", model or LinearRegression())])


X = ames[NUMERIC + CATEGORICAL]
width = pipeline().fit(X, y).named_steps["prep"].transform(X).shape[1]
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical columns "
      f"-> {width} columns once the categories are one-hot encoded")

if ".venv" not in sys.executable:
    print("⚠ not the course environment (the uv .venv) — numbers may differ slightly from class")

# Warm-up data, defined HERE so its body is not on the projector before the reveal
# (in the student notebook it sits inline -- laptops are closed for the warm-up).

def candidates(seed, n=100, p=50):
    """n observations of p candidate predictors and an outcome."""
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(size=(n, p)), columns=[f"x{i:02d}" for i in range(p)])
    return X, pd.Series(rng.normal(size=n), name="y")


print(f"ready — {len(ames):,} houses")

# Recap: what Thursday's studio found

## `course/reference/ref_day06_studio.ipynb` — worked answers to Thursday's three questions

Thursday asked whether shuffled cross-validation overstates how well a model will predict
**later** data. The reference checks it three ways: a third model, dropping the `yr`
column, and Ames by month — and compares every gap against how much random splits move
on their own.

### The sentence to take away: a gap is only big relative to the noise, and a check is only reassuring relative to what it could have seen.

# Exchangeable — the technical version

Rows $Z_1, \dots, Z_n$ are **exchangeable** if shuffling them does not change their joint distribution:

$$(Z_1, \dots, Z_n) \;\overset{d}{=}\; (Z_{\pi(1)}, \dots, Z_{\pi(n)}) \quad \text{for every reordering } \pi$$

Here $Z_i = (X_i, Y_i)$ is a whole row. In words: **the order of the rows carries no information.**

- **i.i.d. ⇒ exchangeable**, but not the reverse. Drawing without replacement is exchangeable but not independent.
- For honest evaluation, **the rows you will predict must be exchangeable with the rows you trained on.** That is what a shuffled fold assumes.
- **Time order** breaks it. A row from 2012 is not interchangeable with one from 2011.
- **The same unit on both sides of the split** breaks it. The next row you predict is a *new* patient; your test rows are patients the model has already seen.

### You cannot prove it from one dataset. You can only test what it implies — and Thursday's forward split was one such test.

# Lab 3 in tonight · Lab 4 out today

## **Lab 3** — due tonight, 11:59 pm

## **Lab 4** — due Tuesday Oct 6, 11:59 pm

### `git pull` first. Restart & Run All. Upload the file you ran.

# Warm-up: choosing predictors by p-value

## A hundred observations. Fifty candidate predictors. One outcome.

A common way to decide which predictors belong in a regression:

### Forward stepwise selection

1. Start with **no predictors**.
2. For every predictor not yet in the model, fit OLS with the current predictors
   **plus that one**, and read off its p-value.
3. **Add the one with the smallest p-value** — if it is below 0.05.
4. Go back to 2. **Stop** when no remaining predictor gets below 0.05.

### Then report the model it ends with.

# How each p-value in that procedure is computed

## The usual t-test for that one coefficient, in the model that includes it

## $t = \hat\beta \,/\, \mathrm{SE}(\hat\beta)$, on $n - (k+1)$ degrees of freedom

### $k$ = predictors in that model, counting the new one; $+1$ for the intercept.
### At step 1: $100 - 2 = 98$.

### It answers: *if this coefficient were really zero, how often would $|t|$ come out this large?*

### Smallest p-value = largest $|t|$ = the biggest drop in the residual sum of squares.

In [ ]:
def forward_stepwise(X, y, alpha=0.05):
    """Add the predictor with the smallest p-value until none is below alpha."""
    chosen = []
    while True:
        best, best_p = None, alpha
        for c in X.columns.difference(chosen):
            p = sm.OLS(y, sm.add_constant(X[chosen + [c]])).fit().pvalues[c]
            if p < best_p:
                best, best_p = c, p
        if best is None:
            return chosen
        chosen.append(best)


Xc, yc = candidates(7)
chosen = forward_stepwise(Xc, yc)
fit = sm.OLS(yc, sm.add_constant(Xc[chosen])).fit()
print(pd.DataFrame({"coef": fit.params[chosen], "p-value": fit.pvalues[chosen]})
      .round(4).to_string())
print(f"\n  R-squared {fit.rsquared:.3f}    overall F-test p = {fit.f_pvalue:.2g}")
print(f"  OLS fits it ran to choose them: {sum(50 - i for i in range(len(chosen) + 1))}")

# Which of these would you put in the abstract?

## Stepwise selection on those 100 observations ended with this model:

## Five predictors. Every p-value below 0.03.
## Overall F-test p = 1.5e-07.

# Every column was random noise.
## So was the outcome.

### To choose those five it ran **285** tests: 50, then 49, 48, 47, 46 — and 45 that found nothing.
### Fifty null tests at step 1: the chance at least one lands below 0.05 is about $1 - 0.95^{50} = 0.92$.

### It found "significant" predictors anyway, because it ran hundreds of tests and kept the winners.

### Is this a fluke? Run stepwise on twenty different noise datasets.
### Then test the predictors it chose on brand-new rows it never saw — are they still significant?

In [ ]:
runs = []
for seed in range(20):
    Xc, yc = candidates(seed)
    chosen = forward_stepwise(Xc, yc)
    X_new, y_new = candidates(1000 + seed)                # fresh rows, same kind of noise
    again = (sm.OLS(y_new, sm.add_constant(X_new[chosen])).fit().pvalues[chosen] < 0.05
             if chosen else pd.Series(dtype=bool))
    runs.append({"chosen": len(chosen),
                 "R2": sm.OLS(yc, sm.add_constant(Xc[chosen])).fit().rsquared if chosen else 0.0,
                 "significant again": int(again.sum())})
runs = pd.DataFrame(runs)

print(f"  runs that found at least one 'significant' predictor   {(runs.chosen > 0).sum()} of 20")
print(f"  median predictors chosen {runs.chosen.median():.0f}, median R-squared {runs.R2.median():.3f}")
print(f"  chosen predictors significant again on fresh rows      "
      f"{runs['significant again'].sum()} of {runs.chosen.sum()}")
print(f"  what chance alone gives (5% of them)                   "
      f"{0.05 * runs.chosen.sum():.1f}")
print(f"  chance 50 null tests include one below 0.05             {1 - 0.95 ** 50:.2f}")

# Choosing on the data you report is Meeting 1's mistake again

## **Meeting 1** — you *scored* a model on the data you fit it to, and the score flattered it.

## **Today** — you *chose* predictors on the data you then report, and the p-values flattered them.

### Each p-value in that table is right for a predictor chosen in advance.
### None was — each was the smallest of up to fifty.

### Whatever made the choice cannot also be the evidence for it.

# What happens to least squares with many columns and few rows?

## Ames, every column — 319 once the categories are one-hot encoded.

## Fit OLS, ridge and lasso on 2,197 houses, then on only 300, and score each on held-out houses.

### Ten random splits at each size, not one.

### Predict first: what happens to OLS at 300 houses?

In [ ]:
rows = []
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=733, random_state=seed)
    for n in (2197, 300):
        for name, model in [("OLS", LinearRegression()),
                            ("ridge", Ridge(alpha=10)),
                            ("lasso", Lasso(alpha=300, max_iter=20_000))]:
            fit = pipeline(model).fit(X_train[:n], y_train[:n])
            rows.append({"houses": n, "model": name,
                         "train R2": r2_score(y_train[:n], fit.predict(X_train[:n])),
                         "test R2": r2_score(y_test, fit.predict(X_test)),
                         "test MAE $": mean_absolute_error(y_test, fit.predict(X_test))})
brake = pd.DataFrame(rows)

print("  medians over 10 splits, and the worst test R-squared of the 10")
print(brake.groupby(["houses", "model"], sort=False)
      .agg(train_R2=("train R2", "median"), test_R2=("test R2", "median"),
           worst_test_R2=("test R2", "min"), test_MAE=("test MAE $", "median"))
      .round(3).to_string())

# A penalty helps exactly when the data cannot pin the coefficients down

Regularization adds a penalty — a brake — to least squares. What it was worth, in held-out R²:

## **2,197 houses** — worth nothing. All three within a hundredth.

## **300 houses** — worth everything. OLS 0.384, ridge 0.841.

### A brake is worth exactly as much as the data
### cannot pin the coefficients down.

### That is why ridge matched OLS on the bike data: 17,379 rows and 12 columns is the 2,197-house case, only more so.

## This has a name: **overfitting**

### Learning noise particular to the sample as if it were signal.
### You see it as training performance far above held-out: OLS on 300 houses, **0.983** against **0.384**.
### It depends on flexibility against data — and a penalty is the brake on it.

# Ridge and lasso: least squares plus a charge for large coefficients

## Both minimize the usual residual sum of squares **plus** a charge on the size of the coefficients. α sets the price; α = 0 is plain OLS.

| | minimizes | the charge |
|---|---|---|
| **ridge** | RSS + α · Σ βⱼ² | squares — big ones expensive, small ones nearly free |
| **lasso** | RSS + α · Σ \|βⱼ\| | absolute values — every coefficient pays the same rate |

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=733, random_state=0)
X300, y300 = X_train[:300], y_train[:300]

dial = []
for a_ridge, a_lasso in [(0.1, 3), (1, 30), (10, 300), (100, 3000), (1000, 30000)]:
    r = pipeline(Ridge(alpha=a_ridge)).fit(X300, y300)
    l = pipeline(Lasso(alpha=a_lasso, max_iter=20_000)).fit(X300, y300)
    dial.append({"ridge α": a_ridge,
                 "ridge: size of coefficients": np.linalg.norm(r.named_steps["model"].coef_),
                 "ridge MAE $": mean_absolute_error(y_test, r.predict(X_test)),
                 "lasso α": a_lasso,
                 "lasso: nonzero of 319": int((l.named_steps["model"].coef_ != 0).sum()),
                 "lasso MAE $": mean_absolute_error(y_test, l.predict(X_test))})
print(pd.DataFrame(dial).round({"ridge: size of coefficients": 0, "ridge MAE $": 0,
                                 "lasso MAE $": 0}).to_string(index=False))

# Ridge shrinks every coefficient; the lasso sets some exactly to zero

## On the same 300 houses, as α grows: ridge pulls all 319 coefficients toward zero but none reaches it; the lasso drops them one at a time, from 230 nonzero down to 4.

## Held-out error is U-shaped in α. Too little penalty: the collapse you just saw. Too much: the model fits nothing.

### **Which α?** — Thursday. It is harder than it looks.

# A penalty depends on the units of your columns; least squares does not

## Meeting 4: standardizing the columns before OLS changed its predictions by exactly `0.000000` — OLS just rescales each coefficient to compensate.

## A penalty charges for the **size** of a coefficient —
## and size depends on units. The same effect is a tiny coefficient per square foot and a large one per acre, so the penalty charges them differently.

In [ ]:
FIVE = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
X5_train, X5_test, y5_train, y5_test = train_test_split(
    ames[FIVE], y, test_size=0.25, random_state=764)

for name, model in [("OLS", LinearRegression()), ("ridge", Ridge(alpha=10)),
                    ("lasso", Lasso(alpha=1000))]:
    scaled = pipeline(model, FIVE, [], scale=True).fit(X5_train, y5_train).predict(X5_test)
    raw = pipeline(model, FIVE, [], scale=False).fit(X5_train, y5_train).predict(X5_test)
    print(f"  {name:<6} largest change in any prediction when the scaler is removed: "
          f"${np.abs(scaled - raw).max():,.6f}")

# Experiment: change only the units of one column

## `Lot_Area` in square feet, then in acres — same houses, same information.

## Fit OLS, ridge and lasso, each with and without the scaler.

### Predict first: which pipelines' predictions change?

In [ ]:
for unit, factor in [("square feet", 1), ("acres", 1 / 43_560)]:
    X_unit = X5_train.copy()
    X_unit["Lot_Area"] = X_unit["Lot_Area"] * factor
    for scale in (True, False):
        lasso = pipeline(Lasso(alpha=1000), FIVE, [], scale=scale).fit(X_unit, y5_train)
        coef = lasso.named_steps["model"].coef_[FIVE.index("Lot_Area")]
        print(f"  Lot_Area in {unit:<12} scaler {'on ' if scale else 'off'}   "
              f"lasso coefficient {coef:>12.4f}")

# With a penalty, standardizing changes the answer

## Without the scaler, the lasso dropped `Lot_Area` in acres — in acres it needs a large coefficient, and the penalty made that too expensive. The model decided lot size does not matter because of the units somebody typed.

## With the scaler, every column is measured in standard deviations, so the units you typed no longer matter.

### The scaler estimates means and SDs from the data, so it still goes inside the pipeline.

# Studio: how much does the lasso's choice of columns depend on luck and on units?

## `course/Day07_092926/` — 36 numeric columns, 300 houses, your own sample

### 1. Plot the lasso paths. Last column standing? How many left at α = 3000?
###    Same for ridge — does any coefficient reach exactly zero?
### 2. Put `Lot_Area` in acres. Refit, scaler on and off. Who noticed?
### 3. Ten different samples. How often is each column kept?
###    **Write your prediction down first.**

# Halfway

## Path plot done?

## Q3 prediction written down? Put it on the board.

# How often is each column kept?

## The lasso on ten different samples of 300 houses. A column kept in all ten is a stable finding; one that comes and goes is not.

In [ ]:
kept = []
for s in range(10):
    draw = ames.sample(300, random_state=s)
    coef = (pipeline(Lasso(alpha=3000, max_iter=10_000), NUMERIC, [])
            .fit(draw[NUMERIC], draw["SalePrice"]).named_steps["model"].coef_)
    kept.append(coef != 0)
kept = pd.DataFrame(kept, columns=NUMERIC)
times = kept.sum().sort_values(ascending=False)

print(f"  columns kept, sample by sample: {kept.sum(axis=1).tolist()}")
print(f"  kept in all ten        {(times == 10).sum():>3}   {times[times == 10].index.tolist()}")
print(f"  kept in some, not all  {((times > 0) & (times < 10)).sum():>3}")
print(f"  never kept             {(times == 0).sum():>3}")

# Compare

## 3. How many survived all ten — and what did you predict?

## 1. Last column standing on your sample?

## 2. Did any unscaled pipeline keep `Lot_Area` in acres?

# Exit ticket

## The lasso kept about eighteen columns on your sample — and you have just seen how much that list changes from sample to sample.

## Write one sentence about which features "matter"
## that you could defend — and one you would refuse to write.

# Before Thursday

## **Lab 3** due tonight · **Lab 4** due Tuesday Oct 6

## Thursday: **which α?** — tuning without contaminating
## the number you are tuning against.

## Reading: ISLP §6.2.3